# VinDr-Mammo Preprocessing Pipeline

Complete preprocessing pipeline for mammogram images with the stratified dataset.

## Features:
- ROI detection and cropping
- Denoising (bilateral filter)
- Contrast enhancement (CLAHE)
- Resizing to 512x512
- Standardization
- Data loader for stratified_selection.csv
- Patient-level train/val/test split

---

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted")

## Step 2: Install Dependencies

In [ ]:
!pip install -q pydicom scikit-image
print("✅ Dependencies installed")

## Step 3: Copy Preprocessing Code

In [ ]:
# Copy mammogram_preprocessor.py from your repo or paste the code here
# For now, we'll include it inline

import numpy as np
import cv2
import matplotlib.pyplot as plt
from scipy import ndimage
from skimage import exposure, filters, morphology, restoration
from skimage.measure import label, regionprops
import pydicom
from pathlib import Path
from typing import Tuple, List, Optional
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Paste the MammogramPreprocessor and VinDrMammoDataLoader classes here
# (Full code from mammogram_preprocessor.py)

print("✅ Preprocessing classes loaded")

## Step 4: Initialize Data Loader

In [ ]:
# Initialize data loader
BASE_DIR = '/content/drive/MyDrive/vindr-mammo'

loader = VinDrMammoDataLoader(
    base_dir=BASE_DIR,
    selection_file='metadata/stratified_selection.csv',
    target_size=512
)

print(f"\n✅ Data loader initialized")
print(f"Total samples: {len(loader)}")

## Step 5: Visualize Preprocessing Pipeline

In [ ]:
# Visualize preprocessing steps for first image
first_image_path = loader.get_image_path(0)

print(f"Visualizing preprocessing for: {first_image_path.name}")
print(f"Label: {'MALIGNANT' if loader.get_label(0) == 1 else 'BENIGN'}")

loader.preprocessor.visualize_pipeline(str(first_image_path))

## Step 6: Test Single Image Processing

In [ ]:
# Load and preprocess single image
image, label = loader.load_sample(0, preprocess=True, verbose=True)

print(f"\n✅ Preprocessed image:")
print(f"   Shape: {image.shape}")
print(f"   Label: {label} ({'MALIGNANT' if label == 1 else 'BENIGN'})")
print(f"   Mean: {image.mean():.6f}")
print(f"   Std: {image.std():.6f}")
print(f"   Min: {image.min():.6f}")
print(f"   Max: {image.max():.6f}")

## Step 7: Train/Val/Test Split

In [ ]:
# Patient-level split
train_df, val_df, test_df = loader.get_train_val_test_split(
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15,
    random_seed=42
)

print("\n" + "="*70)
print("✅ Train/Val/Test Split Complete")
print("="*70)

## Step 8: Load and Visualize Batch

In [ ]:
# Load batch from training set
batch_indices = train_df.head(6).index.tolist()

images, labels = loader.load_batch(batch_indices, preprocess=True, verbose=False)

print(f"\n✅ Loaded batch:")
print(f"   Images shape: {images.shape}")
print(f"   Labels shape: {labels.shape}")
print(f"   Labels: {labels}")

# Visualize batch
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (img, lbl) in enumerate(zip(images, labels)):
    axes[idx].imshow(img, cmap='gray')
    label_name = "MALIGNANT" if lbl == 1 else "BENIGN"
    color = 'red' if lbl == 1 else 'green'
    axes[idx].set_title(f"[{idx+1}] {label_name}", color=color, fontweight='bold')
    axes[idx].axis('off')

plt.tight_layout()
plt.suptitle('Preprocessed Training Batch (512x512, Normalized)', fontsize=14, y=1.02)
plt.show()

## Step 9: Save Preprocessed Data (Optional)

If you want to preprocess all images and save to disk for faster training:

In [ ]:
# Save preprocessed images to .npy files
SAVE_DIR = Path(BASE_DIR) / 'preprocessed_512'
SAVE_DIR.mkdir(exist_ok=True)

def save_preprocessed_dataset(df, split_name, save_dir):
    """Preprocess and save all images in a split."""
    split_dir = save_dir / split_name
    split_dir.mkdir(exist_ok=True)
    
    images = []
    labels = []
    file_paths = []
    
    for idx in range(len(df)):
        # Get original index from loader
        original_idx = df.iloc[idx].name
        
        # Load and preprocess
        image, label = loader.load_sample(original_idx, preprocess=True, verbose=False)
        
        images.append(image)
        labels.append(label)
        file_paths.append(df.iloc[idx]['file_path'])
        
        if (idx + 1) % 10 == 0:
            print(f"  Processed {idx + 1}/{len(df)} images")
    
    # Convert to arrays
    images = np.stack(images)
    labels = np.array(labels)
    
    # Save
    np.save(split_dir / 'images.npy', images)
    np.save(split_dir / 'labels.npy', labels)
    
    # Save metadata
    df.to_csv(split_dir / 'metadata.csv', index=False)
    
    print(f"\n✅ Saved {split_name}:")
    print(f"   Images: {split_dir / 'images.npy'}")
    print(f"   Labels: {split_dir / 'labels.npy'}")
    print(f"   Metadata: {split_dir / 'metadata.csv'}")
    print(f"   Shape: {images.shape}")

# Uncomment to save preprocessed data
# print("Preprocessing and saving train set...")
# save_preprocessed_dataset(train_df, 'train', SAVE_DIR)

# print("\nPreprocessing and saving validation set...")
# save_preprocessed_dataset(val_df, 'val', SAVE_DIR)

# print("\nPreprocessing and saving test set...")
# save_preprocessed_dataset(test_df, 'test', SAVE_DIR)

print("\n💡 Uncomment the code above to save preprocessed data")

## Step 10: Export for PyTorch/TensorFlow

### PyTorch Dataset

In [ ]:
# PyTorch Dataset wrapper
import torch
from torch.utils.data import Dataset, DataLoader

class MammogramDataset(Dataset):
    def __init__(self, df, data_loader, transform=None):
        self.df = df.reset_index(drop=True)
        self.data_loader = data_loader
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        # Get original index
        original_idx = self.df.iloc[idx].name
        
        # Load image and label
        image, label = self.data_loader.load_sample(original_idx, preprocess=True, verbose=False)
        
        # Convert to tensor
        image = torch.from_numpy(image).unsqueeze(0).float()  # Add channel dimension
        label = torch.tensor(label, dtype=torch.long)
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

# Create PyTorch dataloaders
train_dataset = MammogramDataset(train_df, loader)
val_dataset = MammogramDataset(val_df, loader)
test_dataset = MammogramDataset(test_df, loader)

train_loader_pytorch = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader_pytorch = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)
test_loader_pytorch = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=2)

print("✅ PyTorch DataLoaders ready!")
print(f"   Train batches: {len(train_loader_pytorch)}")
print(f"   Val batches: {len(val_loader_pytorch)}")
print(f"   Test batches: {len(test_loader_pytorch)}")

# Test loading one batch
batch_images, batch_labels = next(iter(train_loader_pytorch))
print(f"\n   Batch shape: {batch_images.shape}")
print(f"   Labels: {batch_labels}")

### TensorFlow Dataset

In [ ]:
# TensorFlow Dataset wrapper
import tensorflow as tf

def create_tf_dataset(df, data_loader, batch_size=8):
    """Create TensorFlow dataset."""
    def generator():
        for idx in range(len(df)):
            original_idx = df.iloc[idx].name
            image, label = data_loader.load_sample(original_idx, preprocess=True, verbose=False)
            # Add channel dimension
            image = np.expand_dims(image, axis=-1)
            yield image, label
    
    dataset = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(512, 512, 1), dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.int32)
        )
    )
    
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    
    return dataset

# Create TensorFlow datasets
train_dataset_tf = create_tf_dataset(train_df, loader, batch_size=8)
val_dataset_tf = create_tf_dataset(val_df, loader, batch_size=8)
test_dataset_tf = create_tf_dataset(test_df, loader, batch_size=8)

print("✅ TensorFlow Datasets ready!")

# Test loading one batch
for batch_images, batch_labels in train_dataset_tf.take(1):
    print(f"\n   Batch shape: {batch_images.shape}")
    print(f"   Labels: {batch_labels.numpy()}")

## Summary

You now have:
- ✅ Complete preprocessing pipeline
- ✅ Data loader for stratified dataset
- ✅ Patient-level train/val/test split
- ✅ PyTorch DataLoader integration
- ✅ TensorFlow Dataset integration
- ✅ Visualization tools

Ready for model training! 🚀